# 🚜 PAVI Core: Unified Road Inspection Pipeline (v3.0)
### Professional-Grade Pothole & Crack Analysis | YOLOv11 + ByteTrack + VLM Auditor + SAM 2

---

**Overview:**
This pipeline represents the high-precision backend for the PAVI dashboard. It integrates temporal tracking to prevent duplicate counts, VLM semantic validation to eliminate false positives, and Inverse Perspective Mapping (IPM) for sub-centimeter real-world measurements.

**Core Subsystems:**
1.  **Temporal Hub:** YOLOv11 + ByteTrack for consistent object identification.
2.  **Semantic Auditor:** VLM (GPT-4o/Qwen) validates high-quality crops *once* per unique Track ID.
3.  **Geometric Engine:** Interactive Homography (A4-based) or Geometric IPM for area/length calculation.
4.  **Segmentation Hub:** SAM 2 (Segment Anything 2) for pixel-perfect boundary extraction.

## 1️⃣ Global Configuration & Environment

In [ ]:
# @title 🚀 1.1 Install Dependencies & Setup Environment
import os
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# CORE
!pip install -q -U ultralytics
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
!pip install -q opencv-python pandas numpy tqdm scikit-image openai

# FETCH SAM 2 WEIGHTS
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

print("✅ Toolchain deployment complete.")

In [ ]:
# @title ⚙️ 1.2 Pipeline Parameters
import os

# @markdown ### 📂 Task & Mode
TASK_TYPE = "Potholes" # @param ["Potholes", "Cracks"]
INFERENCE_MODE = "Segmentation" # @param ["Detection", "Segmentation"]
VLM_VALIDATION = True # @param {type:"boolean"}

# @markdown ### 🔗 File Paths
VIDEO_PATH = "/content/video.mp4" # @param {type:"string"}
GPS_LOG_PATH = "/content/gps_log.csv" # @param {type:"string"}
DET_MODEL_PATH = "/content/best_model.pt" # @param {type:"string"}
CALIB_IMAGE_PATH = "/content/calibration.jpg" # @param {type:"string"}

# @markdown ### 🤖 VLM Configuration
VLM_API_KEY = "" # @param {type:"string"}
VLM_MODEL = "gpt-4o-mini" # @param ["gpt-4o-mini", "qwen2.5-vl"]

# @markdown ### 📏 Measurement Params
ROAD_WIDTH_M = 3.5 # @param {type:"number"}
MANUAL_CALIB_POINTS = [] # @param {type:"raw"}

print(f"✅ Configured for {TASK_TYPE} in {INFERENCE_MODE} mode.")

## 2️⃣ Advanced Measurement Engine (Geometric Core)

In [ ]:
import cv2
import numpy as np
from skimage.morphology import skeletonize

def order_points(pts):
    """Orders coordinates: top-left, top-right, bottom-right, bottom-left."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0], rect[2] = pts[np.argmin(s)], pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1], rect[3] = pts[np.argmin(diff)], pts[np.argmax(diff)]
    return rect

def calculate_homography_and_scale(calibration_image_path, manual_points=None, road_width_m=3.5):
    """Calculates Homography and Pixel-to-Meter scale using A4 calibration paper."""
    if not os.path.exists(calibration_image_path): return None, None
    img = cv2.imread(calibration_image_path)
    if img is None: return None, None
    
    rect = None
    if manual_points and len(manual_points) == 4:
        rect = order_points(np.array(manual_points, dtype="float32"))
    else:
        # Auto-detect A4
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        thresh = cv2.bitwise_not(thresh)
        cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
        for c in cnts:
            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.02 * peri, True)
            if len(approx) == 4:
                rect = order_points(approx.reshape(4, 2))
                break
    
    if rect is None: return None, None
    
    scale = 1000.0 # 1000 pixels per meter
    dst = np.array([[0,0], [0.21*scale, 0], [0.21*scale, 0.297*scale], [0, 0.297*scale]], dtype="float32")
    H = cv2.getPerspectiveTransform(rect, dst)
    return H, scale

def measure_defect(mask, width, height, bbox=None, H=None, scale=1000.0, task_type="Potholes"):
    """
    Unified measurement tool.
    Potholes -> Area (m²)
    Cracks -> Length (m)
    """
    # Clean mask inside bbox
    clean_mask = np.zeros_like(mask, dtype=bool)
    if bbox is not None:
        x1, y1, x2, y2 = map(int, bbox)
        clean_mask[y1:y2, x1:x2] = mask[y1:y2, x1:x2] > 0.0
    else:
        clean_mask = mask > 0.0

    if task_type == "Cracks":
        skeleton = skeletonize(clean_mask)
        pixel_val = np.sum(skeleton)
        if H is not None:
            val_m = pixel_val / scale 
        else:
            ppm = (width * 0.8) / ROAD_WIDTH_M
            val_m = pixel_val / ppm
        return val_m, (skeleton, clean_mask)
    
    else: # Potholes
        if H is not None:
            m_u8 = (clean_mask).astype(np.uint8) * 255
            warped = cv2.warpPerspective(m_u8, H, (width*2, height*2))
            val_m = cv2.countNonZero(warped) / (scale ** 2)
        else:
            ppm = (width * 0.8) / ROAD_WIDTH_M
            val_m = np.sum(clean_mask) / (ppm ** 2)
        return val_m, clean_mask

## 3️⃣ VLM Auditor (Semantic Validation Engine)

In [ ]:
import base64
from openai import OpenAI
import cv2

class VLMAuditor:
    def __init__(self, api_key, model="gpt-4o-mini"):
        self.client = OpenAI(api_key=api_key) if api_key else None
        self.model = model

    def encode_image(self, image):
        _, buffer = cv2.imencode(".jpg", image)
        return base64.b64encode(buffer).decode("utf-8")

    def audit_defect(self, crop, defect_type="pothole"):
        """
        Asks the VLM for a 'Second Opinion' to eliminate false positives.
        """
        if not self.client:
            return True # Pass-through if no API key
            
        base64_image = self.encode_image(crop)
        prompt = f"Inspect this road crop. Is there a clearly visible {defect_type}? Reply only YES or NO."
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                        ],
                    }
                ],
                max_tokens=5
            )
            answer = response.choices[0].message.content.strip().upper()
            return "YES" in answer
        except Exception as e:
            print(f"⚠️ VLM Audit Error: {e}")
            return True # Fallback to trust YOLO

## 4️⃣ Core Pipeline Execution

In [ ]:
from ultralytics import YOLO
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from tqdm import tqdm
import pandas as pd

def run_production_pipeline():
    # 1. Initialization
    device = "cuda" if torch.cuda.is_available() else "cpu"
    yolo = YOLO(DET_MODEL_PATH)
    auditor = VLMAuditor(VLM_API_KEY, VLM_MODEL)
    
    predictor = None
    if INFERENCE_MODE == "Segmentation":
        sam2 = build_sam2("sam2_hiera_s.yaml", "sam2_hiera_small.pt", device=device)
        predictor = SAM2ImagePredictor(sam2)

    cap = cv2.VideoCapture(VIDEO_PATH)
    H_matrix, H_scale = calculate_homography_and_scale(CALIB_IMAGE_PATH, MANUAL_CALIB_POINTS, ROAD_WIDTH_M)
    
    track_history = {}
    final_records = []
    validated_ids = set()
    rejected_ids = set()
    
    pbar = tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # Detection + Tracking
        results = yolo.track(frame, persist=True, verbose=False)[0]
        
        if results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            ids = results.boxes.id.cpu().numpy().astype(int)
            confs = results.boxes.conf.cpu().numpy()
            
            for box, tid, conf in zip(boxes, ids, confs):
                if tid in rejected_ids: continue
                
                # VLM Audit (Audit once per ID when confidence and size are optimal)
                if VLM_VALIDATION and tid not in validated_ids:
                    if tid not in track_history: track_history[tid] = []
                    track_history[tid].append({'box': box, 'frame': frame.copy(), 'conf': conf})
                    
                    # picking best crop after 15 frames
                    if len(track_history[tid]) >= 15: 
                        best = max(track_history[tid], key=lambda x: x['conf'])
                        x1, y1, x2, y2 = map(int, best['box'])
                        crop = best['frame'][y1:y2, x1:x2]
                        
                        if auditor.audit_defect(crop, TASK_TYPE.lower()):
                            validated_ids.add(tid)
                            print(f"✅ ID {tid} validated by VLM.")
                        else:
                            rejected_ids.add(tid)
                            print(f"❌ ID {tid} rejected by VLM.")
                            continue
                
                # Processing
                if not VLM_VALIDATION or tid in validated_ids:
                    measurement = 0
                    if INFERENCE_MODE == "Segmentation":
                        predictor.set_image(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                        m, s, _ = predictor.predict(box=box, multimask_output=False)
                        mask = m[0] if m.ndim == 3 else m
                        measurement, _ = measure_defect(mask, frame.shape[1], frame.shape[0], box, H_matrix, H_scale, TASK_TYPE)
                    
                    final_records.append({
                        'id': tid, 'val': measurement, 'lat': 0, 'lon': 0
                    })
        
        pbar.update(1)
    
    cap.release()
    pbar.close()
    return pd.DataFrame(final_records)

# run_production_pipeline()